## Training dataset, independent dataset split

In [41]:
import pandas as pd

In [42]:
# positive dataset
positive_dataset = pd.read_csv("positive_training_dataset.csv")
positive_dataset.head(2)

,Unnamed: 0,uniprot_id,sequence,site,original_id_site,Site,Peptide_[-6:4],ANN,PSSM,SVM,Consensus,pSer/Thr,located_protein
0,806,H7C5C7,XVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMNKTGFVPSNYVER...,73,P16333_85,73.0,KVKRKPsVPDS,0.853,1.287,1.0,1.047,-,P16333
1,808,A0A5F9CW23,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...,85,P16333_85,85.0,KVKRKPsVPDS,0.853,1.287,1.0,1.047,-,P16333


In [43]:
positive_dataset.columns

Index(['Unnamed: 0', 'uniprot_id', 'sequence', 'site', 'original_id_site',
       'Site', 'Peptide_[-6:4]', 'ANN', 'PSSM', 'SVM', 'Consensus', 'pSer/Thr',
       'located_protein'],
      dtype='object')

In [44]:
positive_dataset = positive_dataset[['uniprot_id', 'sequence', 'site', "located_protein"]]
positive_dataset["label"] = 1
positive_dataset["augmentated from uniprot ID"] = positive_dataset["located_protein"]
positive_dataset.head(2)

,uniprot_id,sequence,site,located_protein,label,augmentated from uniprot ID
0,H7C5C7,XVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMNKTGFVPSNYVER...,73,P16333,1,P16333
1,A0A5F9CW23,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...,85,P16333,1,P16333


In [45]:
# negative dataset
negative_dataset = pd.read_csv("labeled_and_augmented_negative_data.csv")
negative_dataset.head(2)

,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,binned_protein_length,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?
0,0,309110,P51843_S_457,P51843,457,P51843_457,470,0.013821,0.424671,0.044,0,0,1,0,0_0_1_0,0,1,0
1,1,270130,P62324_S_90,P62324,90,P62324_90,171,0.185198,0.427061,0.105,0,0,1,0,0_0_1_0,0,1,0


In [46]:
# add sequence for each protein
# build the uniref90_48_dict
from Bio import SeqIO
import re

def parse_fasta(file_path):
    fasta_dict = {}
    for record in SeqIO.parse(file_path, "fasta"):
        # Extract the Uniprot ID from the description line
        # Typically, the Uniprot ID is the first part of the description by '|' delimiter
        # For headers like ">sp|P12345|... description", use the second part:
        record_id = record.id
        # print(record_id)
        uniprot_id, _ = get_species_uniprot_id_human(record_id)
        # print(uniprot_id)
        # print(record.seq)
        
        # Add to dictionary
        fasta_dict[uniprot_id] = str(record.seq)
    
    return fasta_dict

def get_species_uniprot_id_human(hit_def):
    """
    Extract UniProt ID and taxonomy ID from the hit definition string.
    Assumes hit_def follows the format: 'sp|UniProtID|Description OS=Species OX=TaxID ...'.
    """
    parts = hit_def.split('|')
    
    if len(parts) < 3:
        return None, None

    uniprot_id = parts[1]
    
    # Extract taxonomy ID
    match = re.search(r'OX=(\d+) ', hit_def)
    if match:
        taxid = match.group(1)
    else:
        taxid = None
    return uniprot_id, taxid

In [47]:
# Example usage:
file_path = r'uniprotkb_organism_id_9606_AND_reviewed_2025_02_19.fasta'
human_dict = parse_fasta(file_path)
len(human_dict)

negative_dataset["sequence"] = negative_dataset["uniprot_id"].map(human_dict)
negative_dataset

,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,binned_protein_length,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?,sequence
0,0,309110,P51843_S_457,P51843,457,P51843_457,470,0.013821,0.424671,0.044,0,0,1,0,0_0_1_0,0,1,0,MAGENHQWQGSILYNMLMSAKQTRAAPEAPETRLVDQCWGCSCGDE...
1,1,270130,P62324_S_90,P62324,90,P62324_90,171,0.185198,0.427061,0.105,0,0,1,0,0_0_1_0,0,1,0,MHPFYTRAATMIGEIAAAVSFISKFLRTKGLTSERQLQTFSQSLQE...
2,2,912859,Q86V20_S_67,Q86V20,67,Q86V20_67,835,0.418646,0.251716,0.130,0,0,1,0,0_0_1_0,0,1,0,MSGGSQVHIFWGAPIAPLKITVSEDTASLMSVADPWKKIQLLYSQH...
3,3,1216656,Q96AX2_S_218,Q96AX2,218,Q96AX2_218,223,0.236433,0.466251,0.331,0,0,1,0,0_0_1_0,0,1,0,MTGTPGAVATRDGEAPERSPPCSPSYDLTGKVMLLGDTGVGKTCFL...
4,4,422460,O00462_S_726,O00462,726,O00462_726,879,0.090864,0.322924,0.009,0,0,1,0,0_0_1_0,0,1,0,MRLHLLLLLALCGAGTTAAELSYSLRGNWSICNGNGSLELPGAVPG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2667,2667,1176464,Q92974_S_947,Q92974,947,Q92974_947,986,0.956248,0.621385,0.400,0,1,2,0,0_1_2_0,0,0,0,MSRIESLTRARIDRSRELASKTREKEKMKEAKDARYTNGHLFTTIS...
2668,2668,1182040,Q9NQT8_S_1410,Q9NQT8,1410,Q9NQT8_1410,1826,0.657645,0.365731,0.902,1,1,1,1,1_1_1_1,0,0,0,MGDSKVKVAVRIRPMNRRETDLHTKCVVDVDANKVILNPVNTNLSK...
2669,2669,1209502,P49585_S_331,P49585,331,P49585_331,367,0.653063,0.723660,0.956,0,1,2,1,0_1_2_1,0,0,0,MDAQCSAKVNARKRRKEAPGPNGATEEDGVPSKVQRCAVGLRQPAP...
2670,2670,1212107,P07359_S_130,P07359,130,P07359_130,652,0.078022,0.070325,0.076,0,0,0,0,0_0_0_0,0,0,0,MPLLLLLLLLPSPLHPHPICEVSKVASHLEVNCDKRNLTALPPDLP...


In [48]:
tabel1_df = pd.read_csv("Table 1.csv")
tabel1_df["prot_id_position"] = tabel1_df["human homology accession"] + "_" + tabel1_df["human homology site without letter"].astype(str)
tabel1_df = tabel1_df[tabel1_df["label"]==0]
tabel1_df

,human homology accession,human homology site without letter,human Site and Mutation,label,PMID,prot_id_position
350,O14757,345,S345s,0,"2,014,151,111,333,980",O14757_345
351,Q53ET0,244,S244s,0,"20141511,0",Q53ET0_244
352,P14136,13,S13s,0,"20141511,0",P14136_13
353,P05783,51,S51s,0,"20141511,0",P05783_51
354,Q7Z418,264,S264s,0,"20141511,0",Q7Z418_264
...,...,...,...,...,...,...
940,O75385,660,T660t,0,21819378,O75385_660
941,P40818,718,S718s/P720A,0,17720156,P40818_718
942,P46937,109,S109s,0,"20141511,0",P46937_109
943,P26651,197,S197s,0,11886850,P26651_197


In [49]:
complex_mutations_count = tabel1_df[tabel1_df['human Site and Mutation'].str.contains('/')].shape[0]
complex_mutations_count

30

In [50]:
# map mutation string for each uniqueid 
negative_dataset = pd.merge(
    negative_dataset,
    tabel1_df[['prot_id_position', 'human Site and Mutation']],
    on='prot_id_position',
    how='left'
)
negative_dataset

,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,binned_protein_length,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?,sequence,human Site and Mutation
0,0,309110,P51843_S_457,P51843,457,P51843_457,470,0.013821,0.424671,0.044,0,0,1,0,0_0_1_0,0,1,0,MAGENHQWQGSILYNMLMSAKQTRAAPEAPETRLVDQCWGCSCGDE...,NaN
1,1,270130,P62324_S_90,P62324,90,P62324_90,171,0.185198,0.427061,0.105,0,0,1,0,0_0_1_0,0,1,0,MHPFYTRAATMIGEIAAAVSFISKFLRTKGLTSERQLQTFSQSLQE...,NaN
2,2,912859,Q86V20_S_67,Q86V20,67,Q86V20_67,835,0.418646,0.251716,0.130,0,0,1,0,0_0_1_0,0,1,0,MSGGSQVHIFWGAPIAPLKITVSEDTASLMSVADPWKKIQLLYSQH...,NaN
3,3,1216656,Q96AX2_S_218,Q96AX2,218,Q96AX2_218,223,0.236433,0.466251,0.331,0,0,1,0,0_0_1_0,0,1,0,MTGTPGAVATRDGEAPERSPPCSPSYDLTGKVMLLGDTGVGKTCFL...,NaN
4,4,422460,O00462_S_726,O00462,726,O00462_726,879,0.090864,0.322924,0.009,0,0,1,0,0_0_1_0,0,1,0,MRLHLLLLLALCGAGTTAAELSYSLRGNWSICNGNGSLELPGAVPG...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2676,2667,1176464,Q92974_S_947,Q92974,947,Q92974_947,986,0.956248,0.621385,0.400,0,1,2,0,0_1_2_0,0,0,0,MSRIESLTRARIDRSRELASKTREKEKMKEAKDARYTNGHLFTTIS...,S947s
2677,2668,1182040,Q9NQT8_S_1410,Q9NQT8,1410,Q9NQT8_1410,1826,0.657645,0.365731,0.902,1,1,1,1,1_1_1_1,0,0,0,MGDSKVKVAVRIRPMNRRETDLHTKCVVDVDANKVILNPVNTNLSK...,S1410s
2678,2669,1209502,P49585_S_331,P49585,331,P49585_331,367,0.653063,0.723660,0.956,0,1,2,1,0_1_2_1,0,0,0,MDAQCSAKVNARKRRKEAPGPNGATEEDGVPSKVQRCAVGLRQPAP...,S331s
2679,2670,1212107,P07359_S_130,P07359,130,P07359_130,652,0.078022,0.070325,0.076,0,0,0,0,0_0_0_0,0,0,0,MPLLLLLLLLPSPLHPHPICEVSKVASHLEVNCDKRNLTALPPDLP...,S130s


In [51]:
negative_dataset[negative_dataset["uniprot_id"] == "P26651"]

,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,binned_protein_length,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?,sequence,human Site and Mutation
2520,2513,769666,P26651_S_88,P26651,88,P26651_88,326,0.788093,0.822668,0.878,0,1,3,1,0_1_3_1,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,S88s
2521,2514,769670,P26651_T_95,P26651,95,P26651_95,326,0.816150,0.822668,0.329,0,1,3,0,0_1_3_0,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,T95t
2522,2515,769684,P26651_S_186,P26651,186,P26651_186,326,0.632174,0.822668,0.919,0,1,3,1,0_1_3_1,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,S186s/S188P
2523,2516,769688,P26651_S_197,P26651,197,P26651_197,326,0.779859,0.822668,0.942,0,1,3,1,0_1_3_1,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,S197s
2524,2517,769694,P26651_S_214,P26651,214,P26651_214,326,0.808535,0.822668,0.047,0,1,3,0,0_1_3_0,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,S214s
2525,2518,769707,P26651_S_273,P26651,273,P26651_273,326,0.801317,0.822668,0.831,0,1,3,1,0_1_3_1,0,0,0,MDLTAIYESLLSLSPDVPVPSDHGGTESSPGWGSSGPWSLSPSDSS...,S273s


In [52]:
# map mutation to sequence
def apply_mutation(sequence, mutation_info):
    if isinstance(mutation_info, str):
        mutations = mutation_info.split('/')
        sequence_list = list(sequence)
        count = 0
        for mutation in mutations:
            if len(mutation) >= 3 and mutation[-1].isalpha():
                original_aa = mutation[0]
                position = int(mutation[1:-1]) - 1  # converting 1-based to 0-based index
                mutated_aa = mutation[-1].upper()
    
                if count == 0 and original_aa != mutated_aa:
                    print(f"Warning: first sequence is not phosphorylation for {mutation}")
                    
                
                # Validate the original amino acid at the specified position
                if sequence_list[position] == original_aa:
                    sequence_list[position] = mutated_aa
                else:
                    print(f"Warning: Expected '{original_aa}' at position {position + 1} in sequence, found '{sequence_list[position]}'")
                    print(f"mutation_info: {mutation_info}")
                    
                count += 1
                
        if len(mutations)>=2:
            print(mutation_info)
        
        return ''.join(sequence_list)
    else: 
        return sequence

In [53]:
# Apply mutations
negative_dataset['mutated_sequence'] = negative_dataset.apply(lambda row: apply_mutation(row['sequence'], row['human Site and Mutation']), axis=1)

# Display the DataFrame with the modified sequences
negative_dataset

T181t/P183A
S9s/S7E
S9s/S7s
S367s/T365D
T163t/R160A
T163t/F164A
T163t/P165A
S933s/Q930R
S935s/Q930R/S933s
S1444s/R1441G
S183s/P185A
S259s/S257L
S259s/T260R
S259s/P261S
S259s/P261L
S703s/P705A
S970s/P972A
S378s/S376s
S158s/R155A
mutation_info: S301s/S298D
S301s/S298D
S186s/S188P
S22s/T24t
T24t/S22E
T24t/S22s
S256s/R251A/R252A/R253A
S127s/S128D
T283t/K282R
S718s/P720A


,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,...,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?,sequence,human Site and Mutation,mutated_sequence
0,0,309110,P51843_S_457,P51843,457,P51843_457,470,0.013821,0.424671,0.044,...,0,1,0,0_0_1_0,0,1,0,MAGENHQWQGSILYNMLMSAKQTRAAPEAPETRLVDQCWGCSCGDE...,NaN,MAGENHQWQGSILYNMLMSAKQTRAAPEAPETRLVDQCWGCSCGDE...
1,1,270130,P62324_S_90,P62324,90,P62324_90,171,0.185198,0.427061,0.105,...,0,1,0,0_0_1_0,0,1,0,MHPFYTRAATMIGEIAAAVSFISKFLRTKGLTSERQLQTFSQSLQE...,NaN,MHPFYTRAATMIGEIAAAVSFISKFLRTKGLTSERQLQTFSQSLQE...
2,2,912859,Q86V20_S_67,Q86V20,67,Q86V20_67,835,0.418646,0.251716,0.130,...,0,1,0,0_0_1_0,0,1,0,MSGGSQVHIFWGAPIAPLKITVSEDTASLMSVADPWKKIQLLYSQH...,NaN,MSGGSQVHIFWGAPIAPLKITVSEDTASLMSVADPWKKIQLLYSQH...
3,3,1216656,Q96AX2_S_218,Q96AX2,218,Q96AX2_218,223,0.236433,0.466251,0.331,...,0,1,0,0_0_1_0,0,1,0,MTGTPGAVATRDGEAPERSPPCSPSYDLTGKVMLLGDTGVGKTCFL...,NaN,MTGTPGAVATRDGEAPERSPPCSPSYDLTGKVMLLGDTGVGKTCFL...
4,4,422460,O00462_S_726,O00462,726,O00462_726,879,0.090864,0.322924,0.009,...,0,1,0,0_0_1_0,0,1,0,MRLHLLLLLALCGAGTTAAELSYSLRGNWSICNGNGSLELPGAVPG...,NaN,MRLHLLLLLALCGAGTTAAELSYSLRGNWSICNGNGSLELPGAVPG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2676,2667,1176464,Q92974_S_947,Q92974,947,Q92974_947,986,0.956248,0.621385,0.400,...,1,2,0,0_1_2_0,0,0,0,MSRIESLTRARIDRSRELASKTREKEKMKEAKDARYTNGHLFTTIS...,S947s,MSRIESLTRARIDRSRELASKTREKEKMKEAKDARYTNGHLFTTIS...
2677,2668,1182040,Q9NQT8_S_1410,Q9NQT8,1410,Q9NQT8_1410,1826,0.657645,0.365731,0.902,...,1,1,1,1_1_1_1,0,0,0,MGDSKVKVAVRIRPMNRRETDLHTKCVVDVDANKVILNPVNTNLSK...,S1410s,MGDSKVKVAVRIRPMNRRETDLHTKCVVDVDANKVILNPVNTNLSK...
2678,2669,1209502,P49585_S_331,P49585,331,P49585_331,367,0.653063,0.723660,0.956,...,1,2,1,0_1_2_1,0,0,0,MDAQCSAKVNARKRRKEAPGPNGATEEDGVPSKVQRCAVGLRQPAP...,S331s,MDAQCSAKVNARKRRKEAPGPNGATEEDGVPSKVQRCAVGLRQPAP...
2679,2670,1212107,P07359_S_130,P07359,130,P07359_130,652,0.078022,0.070325,0.076,...,0,0,0,0_0_0_0,0,0,0,MPLLLLLLLLPSPLHPHPICEVSKVASHLEVNCDKRNLTALPPDLP...,S130s,MPLLLLLLLLPSPLHPHPICEVSKVASHLEVNCDKRNLTALPPDLP...


In [55]:
negative_dataset[negative_dataset["human Site and Mutation"] == "S301s/S298D"]

,Unnamed: 0,index,uniqueid,uniprot_id,position,prot_id_position,protein_length,iupred_score,anchor_score,phospho_score,...,binned_iupred_score,binned_anchor_score,binned_phospho_score,stratification_label,label,augmentated data?,homology data?,sequence,human Site and Mutation,mutated_sequence
2504,2497,723156,P55042_S_301,P55042,301,P55042_301,308,0.25031,0.988043,0.937,...,0,3,1,0_0_3_1,0,0,0,MTLNGGGSGAGGSRGGGQERERRRGSTPWGPAPPLHRRSMPVDERD...,S301s/S298D,MTLNGGGSGAGGSRGGGQERERRRGSTPWGPAPPLHRRSMPVDERD...


In [ ]:
negative_dataset = negative_dataset[['uniprot_id', 'sequence', 'position', "stratification_label", 'label']]
negative_dataset["site"] = negative_dataset["position"]
negative_dataset.head(2)

In [ ]:
# Training dataset, independent dataset split

In [ ]:
# check the logic in notebook 1 analysis and figure
# if the uniprot ID have been used in previous dataset, it should be in the training dataset
# if the protein is augmentated from a uniprot ID which are in the previous dataset, it should be in the training dataset

# balance training dataset and independent dataset (80% training and 20% independent dataset)

In [ ]:
import pandas as pd
pred_dataset_name = 'Supplementary_Tables.xlsx'
finder_dataset_name = "NIHMS1881042-supplement-Supplementary_table_dataset_2.xlsx"

In [ ]:
# the data in data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx have 4 tab, combine them into one
file_path = pred_dataset_name
excel_file = pd.ExcelFile(file_path)

# Let's get the sheet names
sheet_names = excel_file.sheet_names

# Create an empty list to hold dataframes
dfs = []

# Loop through the sheet names and read each sheet into a dataframe
for sheet in sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    # Optionally, you can add a column indicating the sheet name, useful for tracking the data source
    df['Source_Sheet'] = sheet
    dfs.append(df)

# Concatenate all the dataframes
pred_dataset_df = pd.concat(dfs, ignore_index=True)
pred_dataset_df = pred_dataset_df[pred_dataset_df['PMID'] != '*Likely NEG sites ']
pred_dataset_df = pred_dataset_df[["Uniprot ID","Site","Residue"]]
pred_dataset_df["Site"] = pred_dataset_df["Site"].astype(int)
pred_dataset_df.head(2)

In [ ]:
# 14-3-3 finder dataset preprocess
finder_dataset_df = pd.read_excel(finder_dataset_name)
finder_dataset_df = finder_dataset_df[["Uniprot ID", 'Site',"Residue"]]
finder_dataset_df.head(2)

In [ ]:
import re
previous_datset_df = pd.concat([pred_dataset_df, finder_dataset_df], ignore_index=True)
previous_datset_df

In [ ]:
# put the uniprot ID into a set
uniprotid_dict = set(previous_datset_df["Uniprot ID"])
len(uniprotid_dict)

In [ ]:
# first, but all the seen uniprot id (previously reported) into the training dataset
positive_training_dataset_df =  positive_dataset[positive_dataset['uniprot_id'].isin(uniprotid_dict)]
positive_training_dataset_df["label"].value_counts()

In [ ]:
remaining_positive_dataset_df =  positive_dataset[~positive_dataset['uniprot_id'].isin(uniprotid_dict)]
remaining_positive_dataset_df.head(2)

In [ ]:
# I have 1330 positive dataset
# 20% will be keep for independent test dataset
training_sample_size = int(1330 * 0.8)
training_sample_size

In [ ]:
# add more positive data into positive_training_dataset_df
# for positive data, add uniprot and augmentated data to about 1200 positive sample

def sample_data(df_source, column_name, target_length):
    # Initialize an empty DataFrame B
    df_target = pd.DataFrame(columns=df_source.columns)
    df_source = df_source.copy()
    # Keep track of the unique values already processed to avoid duplication
    processed_values = set()

    for k_value in positive_training_dataset_df["uniprot_id"]:
        # Get all rows with the same value in the specified column
        rows_with_k = df_source[df_source[column_name] == k_value]

        # Add these rows to the target DataFrame
        df_target = pd.concat([df_target, rows_with_k], ignore_index=True)

        # Update the set of processed values
        processed_values.add(k_value)

        # Drop these rows from the source DataFrame to avoid resampling
        df_source = df_source[df_source[column_name] != k_value]
        

    while len(df_target) < target_length and not df_source.empty:
        # Randomly sample one row from the source DataFrame
        sampled_row = df_source.sample(1)
        k_value = sampled_row.iloc[0][column_name]

        # Check if this value has already been processed
        if k_value not in processed_values:
            # Get all rows with the same value in the specified column
            rows_with_k = df_source[df_source[column_name] == k_value]

            # Add these rows to the target DataFrame
            df_target = pd.concat([df_target, rows_with_k], ignore_index=True)

            # Update the set of processed values
            processed_values.add(k_value)

            # Drop these rows from the source DataFrame to avoid resampling
            df_source = df_source[df_source[column_name] != k_value]

    return df_target


column_name = 'augmentated from uniprot ID'
target_length = training_sample_size - len(positive_training_dataset_df[positive_training_dataset_df['label']==1])
df_sampling_training_positive = sample_data(remaining_positive_dataset_df, column_name, target_length)
df_sampling_training_positive = pd.concat([df_sampling_training_positive, positive_training_dataset_df], ignore_index=True)
df_sampling_training_positive

In [ ]:
# for the training negative data, add to training_sample_size, also make sure the sample following whole distribution
import numpy as np
sample_size = len(df_sampling_training_positive)
df_labeled_and_augmented_negative_data = negative_dataset
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
# first, but all the seen uniprot id (previously reported) into the training dataset
negative_training_dataset_df =  df_labeled_and_augmented_negative_data[df_labeled_and_augmented_negative_data['uniprot_id'].isin(uniprotid_dict)]
negative_values = negative_training_dataset_df['stratification_label'].value_counts(normalize=False)
negative_values

In [ ]:
df_stratum_sample_size_ceil = df_stratum_sample_size_ceil-negative_values
df_stratum_sample_size_ceil = df_stratum_sample_size_ceil.fillna(0).astype(int)
df_stratum_sample_size_ceil

In [ ]:
remaining_negative_dataset_df =  df_labeled_and_augmented_negative_data[~df_labeled_and_augmented_negative_data['uniprot_id'].isin(uniprotid_dict)]
remaining_negative_dataset_df.head(2)

In [ ]:
df_sampling_training_negative = negative_training_dataset_df
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = remaining_negative_dataset_df[remaining_negative_dataset_df["stratification_label"] == index]
        df_tmp = df_tmp[(~df_tmp["uniprot_id"].isin(uniprotid_dict))]
        if len(df_tmp) < value:
            sampled_df = df_tmp
        else:
            sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_training_negative) == 0:
            df_sampling_training_negative = sampled_df
        else:
            df_sampling_training_negative = pd.concat([df_sampling_training_negative, sampled_df], axis=0)

In [ ]:
df_sampling_training_negative

In [ ]:
training_dataset_df = pd.concat([df_sampling_training_positive, df_sampling_training_negative], axis=0)

In [ ]:
training_dataset_df

In [ ]:
training_dataset_df.columns

In [ ]:
training_dataset_df = training_dataset_df[['uniprot_id', 'sequence', 'site', 'label', 'augmentated from uniprot ID']]
training_dataset_df["usage"] = "training"
training_dataset_df

In [ ]:
training_dataset_df["label"].value_counts()

In [ ]:
# all the remaining data are below to indepnedent dataset, about 266 positive and 266 negative sample
# sampling from the remaining negative sample to get 266 negative sample
sample_size = 1330-training_sample_size
df_labeled_and_augmented_negative_data = pd.read_csv("labeled_and_augmented_negative_data.csv")
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
df_sampling_indepnedent_negative = pd.DataFrame()
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = negative_dataset[negative_dataset["stratification_label"] == index]
        df_tmp = df_tmp[(~df_tmp["uniprot_id"].isin(uniprotid_dict))&(~df_tmp["uniprot_id"].isin(set(training_dataset_df["uniprot_id"])))]
        if len(df_tmp) < value:
            sampled_df = df_tmp
        else:
            sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_indepnedent_negative) == 0:
            df_sampling_indepnedent_negative = sampled_df
        else:
            df_sampling_indepnedent_negative = pd.concat([df_sampling_indepnedent_negative, sampled_df], axis=0)

In [ ]:
df_sampling_indepnedent_negative

In [ ]:
# all the remaining positive data as the 300 positive independent dataset
df_sampling_indepnedent_positive = positive_dataset[~positive_dataset['uniprot_id'].isin(df_sampling_training_positive["uniprot_id"])]

In [ ]:
df_sampling_indepnedent_positive

In [ ]:
independent_dataset_df = pd.concat([df_sampling_indepnedent_positive, df_sampling_indepnedent_negative], axis=0)

In [ ]:
independent_dataset_df = independent_dataset_df[['uniprot_id', 'sequence', 'site', 'label', 'augmentated from uniprot ID']]
independent_dataset_df["usage"] = "independent test"
independent_dataset_df

In [ ]:
# ADD features to training and independent dataset
final_dataset = pd.concat([training_dataset_df, independent_dataset_df], axis=0)
final_dataset

In [ ]:
final_dataset.loc[final_dataset["label"] == 0, "augmentated from uniprot ID"] = final_dataset.loc[final_dataset["label"] == 0, "uniprot_id"]
final_dataset

In [ ]:
# Example usage:
file_path = r'uniprotkb_organism_id_9606_AND_reviewed_2025_02_19.fasta'
human_dict = parse_fasta(file_path)
len(human_dict)

mask = final_dataset["sequence"].isna()
final_dataset.loc[mask, "sequence"] = final_dataset.loc[mask, "uniprot_id"].map(human_dict)

In [ ]:
final_dataset

In [ ]:
final_dataset.isna().sum()

In [ ]:
final_dataset.to_csv("final_dataset_with_seq.csv", index=False)

In [ ]:
final_dataset = pd.read_csv("final_dataset_with_seq.csv")
final_dataset